In [11]:
from dotenv import load_dotenv
from sqlalchemy import create_engine, text, bindparam
from datetime import date, datetime, timedelta
from pathlib import Path
import pandas as pd
import os
import urllib3

# Load environment variables from .env file
load_dotenv()

strPresto = ('presto://{username}:{password}@{ipaddress}:{port}/{dbname}/{schema}'
             .format(username=os.getenv('HIVE_SVC_USER'),
                     password=os.getenv('HIVE_SVC_PASS'),
                     ipaddress=os.getenv('HIVE_SVC_ADDRESS'),
                     port=os.getenv('HIVE_SVC_PORT'),
                     dbname=os.getenv('HIVE_SVC_DBNAME'),
                     schema=os.getenv('HIVE_SVC_SCHEMA')))
 
presto_engine = create_engine(strPresto, connect_args={"protocol": "https", "requests_kwargs": {"verify": False}})

# disable certificate warnings
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [12]:
# ------------------------------------------------------------------
# Start date and end dates for dateselectors in queries
# ------------------------------------------------------------------

#start_date = '2025-12-13' 
#end_date = '2026-02-06' 

today = date.today()
diff_to_friday = (4 - today.weekday()) % 7  # Mon=0 ... Fri=4

# Go back 7 weeks since we are counting the current week
eight_weeks_ago = today - timedelta(weeks=7) 

# Find the Saturday of that week (Mon=0 ... Sun=6, Sat=5)
days_since_saturday = (eight_weeks_ago.weekday() - 5) % 7

start_saturday = eight_weeks_ago - timedelta(days=days_since_saturday)
end_friday = today + timedelta(days=diff_to_friday)

#format date as string for Presto query
start_date = start_saturday.strftime('%Y-%m-%d')
end_date = end_friday.strftime('%Y-%m-%d')


In [13]:
start_date

'2025-12-13'

In [14]:
# ------------------------------------------------------------------
# ICP Client list as stored in hive.care.expert_performance_metrics.metric (lowercase)
# ------------------------------------------------------------------

icp_client_list = [
    "pss-verizon",
    "pss-at&t",
    "mob-verizon",
    "mob-at&t"
]

In [15]:
# ------------------------------------------------------------------
# Metric configuration
# metric_list are the metric names as stored in hive.care.expert_performance_metrics.metric (lowercase)
# metric_name_map is optional for pretty display names
# ------------------------------------------------------------------

metric_list = [
    "erp",
    "nsp100",
    "cancellation rate",
    "transfers",
    "resolution rate",
    "crt"
]

metric_name_map = {
    "erp": "ERP",
    "nsp100": "NSP 100",
    "cancellation rate": "Cancel Rate",
    "transfers": "Transfers",
    "resolution rate": "Resolution",
    "crt": "CRT",
}

In [16]:
# ------------------------------------------------------------------
# Observe scorecard list
# observe scorecards are the groupings of behavior scores to be queried from observe data
# ------------------------------------------------------------------

observe_scorecards = {
    "HEROES - Sentiment v1.1",
    "HEROES LITE v.1",
    "HEROES - Solve v1.2",
    "HEROES - Serve v1.2",
    "HEROES - Sell - Smart Offer v1.1",
}

In [17]:
# ------------------------------------------------------------------
# Load SQL template for metric query
# ------------------------------------------------------------------

sql_path = "SQL/epm_expert_data_week.sql"  # <- make sure this path is correct

with open(sql_path, "r") as f:
    METRIC_SQL_TEMPLATE = f.read()

print("Loaded SQL template:")
print(METRIC_SQL_TEMPLATE[:500], "...")

Loaded SQL template:
SELECT 
CAST(week_stop_date AS DATE) AS week_,
--date,
expert_id,
-- year_month,
metric,
icp_client,
tenure_group,
site,
SUM(numerator) AS num,
SUM(denominator) AS den,
ROUND(
COALESCE(
CAST(SUM(numerator) AS DOUBLE) /
NULLIF(CAST(SUM(denominator) AS DOUBLE), 0.0),
0.000
),
3
) AS calc
FROM 
hive.care.expert_performance_metrics a
LEFT OUTER JOIN 
hive.care.l4_asurion_umt_ppx_pay_calendar d ON a."date" = CAST(d.event_date AS DATE)
WHERE 
LOWER(metric) IN :metric_list
AND LOWER(icp_client) IN :icp ...


In [18]:
### Compile SQL With Literal Binds (Code)

#This uses your proven pattern: bind params + `literal_binds=True`.

#python
# ------------------------------------------------------------------
# Build literal SQL for Presto using SQLAlchemy binds
# This allows us to use expanding=True for metric_list and still
# send flattened literal SQL to Presto.
# ------------------------------------------------------------------

def compile_presto_sql(
    sql_template: str,
    engine,
    start_date,
    end_date,
    icp_client_list,
    metric_list,
):
    """
    Creates literal SQL for Presto by binding parameters and compiling
    with literal_binds=True.
    """
    
    stmt = text(sql_template).bindparams(
        bindparam("start_date", value=start_date),
        bindparam("end_date", value=end_date),
        bindparam("icp_client_list", value=list(icp_client_list), expanding=True),
        bindparam("metric_list", value=list(metric_list), expanding=True),
    )

    compiled = stmt.compile(
        engine,
        compile_kwargs={"literal_binds": True}
    )

    return str(compiled)


In [19]:
# ------------------------------------------------------------------
# Query Presto for metrics, client groups, and date range, returning the
# aggregated metrics DataFrame.
# ------------------------------------------------------------------

def query_metrics_presto_group(
    start_date,
    end_date,
    icp_client_list,
    metric_list,
):
    """
    Execute the Presto query for a list of experts over a date range.

    Returns DataFrame with:
      [expert_id, metric, icp_client, site, num, den, calc]
    """

    sql = compile_presto_sql(
        sql_template=METRIC_SQL_TEMPLATE,
        engine=presto_engine,
        start_date=start_date,
        end_date=end_date,
        icp_client_list=icp_client_list,
        metric_list=metric_list,
    )

    # Uncomment to debug generated SQL:
    # print(sql)

    with presto_engine.connect() as conn:
        df = pd.read_sql(sql, conn)

    # Normalize types for downstream joins
    if not df.empty:
        df["icp_client"] = df["icp_client"].astype(str)
        df["metric"] = df["metric"].str.lower()

    return df


In [20]:
df = query_metrics_presto_group(
        start_date,
        end_date,
        icp_client_list,
        metric_list,
    )

In [21]:
df

,week_,expert_id,metric,icp_client,tenure_group,site,num,den,calc
0,2026-01-02,696583,cancellation rate,MOB-Verizon,61-90,falcons,1.0,1.0,1.000
1,2026-01-02,693376,erp,PSS-Verizon,91-120,turtles,100.0,3.0,33.333
2,2026-01-02,684699,crt,PSS-Verizon,121-180,panthers,114637.0,53.0,2162.962
3,2026-01-02,671969,cancellation rate,PSS-Verizon,180+,bears,1.0,4.0,0.250
4,2026-01-02,694912,crt,PSS-Verizon,61-90,raptors,78376.0,60.0,1306.267
...,...,...,...,...,...,...,...,...,...
136975,2025-12-26,697095,nsp100,PSS-Verizon,31-60,panthers,1.0,38.0,0.026
136976,2025-12-19,634530,erp,PSS-Verizon,180+,raptors,100.0,2.0,50.000
136977,2025-12-19,696444,cancellation rate,PSS-Verizon,31-60,panthers,NaN,2.0,0.000
136978,2025-12-19,700122,erp,PSS-Verizon,0-30,bears trn,100.0,1.0,100.000


In [22]:
# Save to CSV in the same directory

from pathlib import Path

file = Path("../data/business_metrics.csv")   # replace with your filename

if file.exists():
    file.unlink()
    df.to_csv("../data/business_metrics.csv", index=False)
else:
    df.to_csv("../data/business_metrics.csv", index=False)
